### Real Time Weather Tracker

The development of a small end-to-end data pipeline that collects real-time weather and air quality data, stores it in a relational database, and answers analytical questions using SQL. The goal is to develop a working, automated system and a portfolio-ready writeup demonstrating practical Python and SQL skills.

#### Background

Real-time data pipelines are a core building block in data-driven roles: pulling data from an external source, storing it reliably, modeling it sensibly, and querying it for insight. This project uses a free, keyless weather API so the focus is entirely on the pipeline and analysis rather than on data access logistics

#### Data Source

The data used in this project is obtained from Open-Meteo (https://open-meteo.com), an open access platform that provides free a weather and air quality API that does not require an APi key and has generous usage limits. it provides;

* Current weather and hourly forecast data

* Historical weather archives (useful for backfilling data early on)

* Air quality data (PM2.5, PM10, ozone, and more)



#### The Tech Stack 

The following tech is used in the project pipeline;

* Python — for API calls and automation (requests, plus scheduling)

* SQL — SQLite to start; optionally PostgreSQL

* pandas — for light data cleaning and reshaping before writing to the database

* Streamlit — for the final interactive dashboard

* matplotlib or seaborn — for building the charts embedded within it

#### Phase 1

The first phase of the project builds a python script that;

1. Calls the Open-Meteo API for each of the chosen cities

2. Extracts the relevant fields from the JSON response (temperature, humidity, wind speed, air quality metrics, etc.)

3. Writes those fields into a SQLite database

**Checkpoint**: Run the script manually several times and confirm new rows appear correctly in the database each time.


#### Importing all essential libraries and initiating the project

In [ ]:
!pip install requests

In [5]:

import sqlite3
from datetime import datetime, timezone

In [3]:
import matplotlib.pyplot as plt
import numpy as np  
import pandas as pd
import requests

Calling the Open-Meteo API for each of the chosen cities

In [6]:
"""
Fetch weather from Open-Meteo for 4 locations and store it in a SQLite database.

Every time you run this script it:
  1. calls the API,
  2. creates a new "run" (with a timestamp) in the database,
  3. inserts the current, hourly and daily data tagged with that run_id,
  4. checks that the expected number of new rows actually appeared.
"""


DB_FILE = "weather.db"
URL = "https://api.open-meteo.com/v1/forecast"

# (name, latitude, longitude) - same order as sent to the API
LOCATIONS = [
    ("Oslo", 59.91, 10.75),
    ("Bergen", 60.39, 5.32),
    ("Trondheim", 63.43, 10.40),
    ("Location 4", 59.93, 9.97),
]

CURRENT_FIELDS = [
    "temperature_2m", "relative_humidity_2m", "apparent_temperature", "is_day",
    "weather_code", "cloud_cover", "pressure_msl", "surface_pressure",
    "precipitation", "rain", "showers", "snowfall", "wind_speed_10m",
    "wind_direction_10m", "wind_gusts_10m",
]
HOURLY_FIELDS = [
    "temperature_2m", "relative_humidity_2m", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "weather_code", "cloud_cover", "uv_index", "sunshine_duration",
]
DAILY_FIELDS = ["sunrise", "sunset", "rain_sum", "uv_index_max"]

# "metno_seamless" = MET Norway first, ECMWF after ~2.5 days.
# Set to None to use Open-Meteo's default "best match".
MODEL = "metno_seamless"

TEXT_FIELDS = {"sunrise", "sunset"}           # everything else is stored as a number
TABLES = ["runs", "current_weather", "hourly_weather", "daily_weather"]


# ---------------------------------------------------------------- API
def fetch_weather():
    params = {
        "latitude": ",".join(str(lat) for _, lat, _ in LOCATIONS),
        "longitude": ",".join(str(lon) for _, _, lon in LOCATIONS),
        "current": ",".join(CURRENT_FIELDS),
        "hourly": ",".join(HOURLY_FIELDS),
        "daily": ",".join(DAILY_FIELDS),
        "timezone": "Europe/Berlin",
    }
    if MODEL:
        params["models"] = MODEL

    response = requests.get(URL, params=params, timeout=20)
    response.raise_for_status()
    data = response.json()
    if isinstance(data, dict):        # a single location comes back as a dict, not a list
        data = [data]
    if len(data) != len(LOCATIONS):
        raise ValueError(f"Expected {len(LOCATIONS)} locations, got {len(data)}")
    return data


# ---------------------------------------------------------------- database setup
def column_defs(fields):
    return ",\n        ".join(
        f"{f} {'TEXT' if f in TEXT_FIELDS else 'REAL'}" for f in fields
    )


def create_tables(conn):
    common = """
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id    INTEGER NOT NULL REFERENCES runs(run_id),
        location  TEXT NOT NULL,
        latitude  REAL,
        longitude REAL,
        time      TEXT NOT NULL,"""

    conn.executescript(f"""
    CREATE TABLE IF NOT EXISTS runs (
        run_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        fetched_at TEXT NOT NULL,          -- UTC time the script ran
        model      TEXT
    );
    CREATE TABLE IF NOT EXISTS current_weather ({common}
        {column_defs(CURRENT_FIELDS)}
    );
    CREATE TABLE IF NOT EXISTS hourly_weather ({common}
        {column_defs(HOURLY_FIELDS)}
    );
    CREATE TABLE IF NOT EXISTS daily_weather ({common}
        {column_defs(DAILY_FIELDS)}
    );
    CREATE INDEX IF NOT EXISTS idx_current_run ON current_weather(run_id);
    CREATE INDEX IF NOT EXISTS idx_hourly_run  ON hourly_weather(run_id);
    CREATE INDEX IF NOT EXISTS idx_daily_run   ON daily_weather(run_id);
    """)


# ---------------------------------------------------------------- inserting
def insert_rows(conn, table, fields, rows):
    cols = ["run_id", "location", "latitude", "longitude", "time", *fields]
    placeholders = ", ".join("?" * len(cols))
    conn.executemany(
        f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({placeholders})", rows
    )
    return len(rows)


def series_rows(base, block, fields):
    """Turn Open-Meteo's column lists (hourly/daily) into one row per time step."""
    times = block["time"]
    columns = [block.get(f) or [None] * len(times) for f in fields]
    return [base + (t,) + tuple(col[i] for col in columns) for i, t in enumerate(times)]


def save_run(conn, data):
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    cur = conn.execute(
        "INSERT INTO runs (fetched_at, model) VALUES (?, ?)",
        (fetched_at, MODEL or "best_match"),
    )
    run_id = cur.lastrowid
    inserted = {"runs": 1, "current_weather": 0, "hourly_weather": 0, "daily_weather": 0}

    for (name, _, _), loc in zip(LOCATIONS, data):
        base = (run_id, name, loc["latitude"], loc["longitude"])

        c = loc["current"]
        current_row = base + (c["time"],) + tuple(c.get(f) for f in CURRENT_FIELDS)
        inserted["current_weather"] += insert_rows(
            conn, "current_weather", CURRENT_FIELDS, [current_row])

        inserted["hourly_weather"] += insert_rows(
            conn, "hourly_weather", HOURLY_FIELDS,
            series_rows(base, loc["hourly"], HOURLY_FIELDS))

        inserted["daily_weather"] += insert_rows(
            conn, "daily_weather", DAILY_FIELDS,
            series_rows(base, loc["daily"], DAILY_FIELDS))

    return run_id, fetched_at, inserted


# ---------------------------------------------------------------- verification
def count_rows(conn):
    return {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in TABLES}


def main():
    data = fetch_weather()            # fetch first, so a failed API call writes nothing

    conn = sqlite3.connect(DB_FILE)
    try:
        create_tables(conn)
        before = count_rows(conn)

        with conn:                    # one transaction: all rows saved, or none
            run_id, fetched_at, inserted = save_run(conn, data)

        after = count_rows(conn)

        print(f"Run #{run_id} saved at {fetched_at} UTC  ->  {DB_FILE}\n")
        print(f"{'table':<16}{'before':>8}{'added':>8}{'after':>8}   check")
        all_ok = True
        for t in TABLES:
            ok = after[t] - before[t] == inserted[t]
            all_ok &= ok
            print(f"{t:<16}{before[t]:>8}{inserted[t]:>8}{after[t]:>8}   "
                  f"{'OK' if ok else 'MISMATCH'}")

        print("\nCurrent weather stored in this run:")
        for row in conn.execute(
            "SELECT location, time, temperature_2m, wind_speed_10m, relative_humidity_2m "
            "FROM current_weather WHERE run_id = ? ORDER BY id", (run_id,)
        ):
            print(f"  {row[0]:<11} {row[1]}  {row[2]} °C, wind {row[3]} km/h, "
                  f"humidity {row[4]} %")

        print(f"\nTotal runs in database: {after['runs']}")
        print("All checks passed." if all_ok else "WARNING: row counts did not match!")
    finally:
        conn.close()


if __name__ == "__main__":
    main()

Run #1 saved at 2026-10-27T12:46:57+00:00 UTC  ->  weather.db

table             before   added   after   check
runs                   0       1       1   OK
current_weather        0       4       4   OK
hourly_weather         0     672     672   OK
daily_weather          0      28      28   OK

Current weather stored in this run:
  Oslo        2026-10-27T13:45  12.5 °C, wind 27.7 km/h, humidity 94.0 %
  Bergen      2026-10-27T13:45  13.5 °C, wind 38.5 km/h, humidity 80.0 %
  Trondheim   2026-10-27T13:45  12.7 °C, wind 26.3 km/h, humidity 67.0 %
  Location 4  2026-10-27T13:45  13.2 °C, wind 20.5 km/h, humidity 94.0 %

Total runs in database: 1
All checks passed.
